<a href="https://colab.research.google.com/github/simonburner/hands-on-LLM/blob/main/ch2/BPE_tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
corpus = [
    "I want to learn how byte pair encoding works "
    "This project follows the process how byte pair encoding works "
    "Let's learn how they work"
]

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [3]:
from collections import defaultdict

In [4]:
word_freqs = defaultdict(int)

for text in corpus:
  words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
  new_words = [word for word, offset in words_with_offsets]
  for word in new_words:
    word_freqs[word] += 1

print(word_freqs)

defaultdict(<class 'int'>, {'I': 1, 'Ġwant': 1, 'Ġto': 1, 'Ġlearn': 2, 'Ġhow': 3, 'Ġbyte': 2, 'Ġpair': 2, 'Ġencoding': 2, 'Ġworks': 2, 'ĠThis': 1, 'Ġproject': 1, 'Ġfollows': 1, 'Ġthe': 1, 'Ġprocess': 1, 'ĠLet': 1, "'s": 1, 'Ġthey': 1, 'Ġwork': 1})


In [5]:
alphabet = []

for word in word_freqs.keys():
  for letter in word:
    if letter not in alphabet:
      alphabet.append(letter)

alphabet.sort()

print(alphabet)

["'", 'I', 'L', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'n', 'o', 'p', 'r', 's', 't', 'w', 'y', 'Ġ']


In [6]:
vocab = ["<|endoftext|>"] + alphabet.copy()

In [7]:
splits = {word: [c for c in word] for word in word_freqs.keys()}

In [8]:
def compute_pair_freqs(splits):
  pair_freqs = defaultdict(int)
  for word, freq in word_freqs.items():
    split = splits[word]
    if len(split) == 1:
      continue
    for i in range(len(split) - 1):
      pair = (split[i], split[i + 1])
      pair_freqs[pair] += freq
  return pair_freqs

In [9]:
pair_freqs = compute_pair_freqs(splits)

for i, key in enumerate(pair_freqs.keys()):
  print(f"{key}: {pair_freqs[key]}")
  if i >= 5:
    break

('Ġ', 'w'): 4
('w', 'a'): 1
('a', 'n'): 1
('n', 't'): 1
('Ġ', 't'): 3
('t', 'o'): 1


In [10]:
best_pair = ""
max_freq = None

for pair, freq in pair_freqs.items():
  if max_freq is None or max_freq < freq:
    best_pair = pair
    max_freq = freq

print(best_pair, max_freq)

('Ġ', 'w') 4


In [11]:
merges = {("Ġ", "w"): "Ġw"}
vocab.append("Ġw")

In [12]:
def merge_pair(a, b, splits):
  for word in word_freqs:
    split = splits[word]
    if len(split) == 1:
      continue
    i = 0
    while i < len(split) - 1:
      if split[i] == a and split[i + 1] == b:
        split = split[:i] + [a + b] + split[i + 2 :]
      else:
        i += 1
    splits[word] = split
  return splits

In [13]:
splits = merge_pair("Ġ", "w", splits)
print(splits["Ġwant"])

['Ġw', 'a', 'n', 't']


In [14]:
vocab_size = 70

while len(vocab) < vocab_size:
  pair_freqs = compute_pair_freqs(splits)
  best_pair = ""
  max_freq = None
  for pair, freq in pair_freqs.items():
    if max_freq is None or max_freq < freq:
      best_pair = pair
      max_freq = freq
  splits = merge_pair(*best_pair, splits)
  merges[best_pair] = best_pair[0] + best_pair[1]
  vocab.append(best_pair[0] + best_pair[1])

In [15]:
print(merges)

{('Ġ', 'w'): 'Ġw', ('o', 'w'): 'ow', ('Ġ', 'p'): 'Ġp', ('Ġ', 't'): 'Ġt', ('Ġ', 'h'): 'Ġh', ('Ġh', 'ow'): 'Ġhow', ('Ġw', 'o'): 'Ġwo', ('Ġwo', 'r'): 'Ġwor', ('Ġwor', 'k'): 'Ġwork', ('Ġ', 'l'): 'Ġl', ('Ġl', 'e'): 'Ġle', ('Ġle', 'a'): 'Ġlea', ('Ġlea', 'r'): 'Ġlear', ('Ġlear', 'n'): 'Ġlearn', ('Ġ', 'b'): 'Ġb', ('Ġb', 'y'): 'Ġby', ('Ġby', 't'): 'Ġbyt', ('Ġbyt', 'e'): 'Ġbyte', ('Ġp', 'a'): 'Ġpa', ('Ġpa', 'i'): 'Ġpai', ('Ġpai', 'r'): 'Ġpair', ('Ġ', 'e'): 'Ġe', ('Ġe', 'n'): 'Ġen', ('Ġen', 'c'): 'Ġenc', ('Ġenc', 'o'): 'Ġenco', ('Ġenco', 'd'): 'Ġencod', ('Ġencod', 'i'): 'Ġencodi', ('Ġencodi', 'n'): 'Ġencodin', ('Ġencodin', 'g'): 'Ġencoding', ('Ġwork', 's'): 'Ġworks', ('Ġp', 'r'): 'Ġpr', ('Ġpr', 'o'): 'Ġpro', ('Ġt', 'h'): 'Ġth', ('Ġth', 'e'): 'Ġthe', ('Ġw', 'a'): 'Ġwa', ('Ġwa', 'n'): 'Ġwan', ('Ġwan', 't'): 'Ġwant', ('Ġt', 'o'): 'Ġto', ('Ġ', 'T'): 'ĠT', ('ĠT', 'h'): 'ĠTh', ('ĠTh', 'i'): 'ĠThi', ('ĠThi', 's'): 'ĠThis', ('Ġpro', 'j'): 'Ġproj', ('Ġproj', 'e'): 'Ġproje'}


In [16]:
print(vocab)

['<|endoftext|>', "'", 'I', 'L', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'n', 'o', 'p', 'r', 's', 't', 'w', 'y', 'Ġ', 'Ġw', 'ow', 'Ġp', 'Ġt', 'Ġh', 'Ġhow', 'Ġwo', 'Ġwor', 'Ġwork', 'Ġl', 'Ġle', 'Ġlea', 'Ġlear', 'Ġlearn', 'Ġb', 'Ġby', 'Ġbyt', 'Ġbyte', 'Ġpa', 'Ġpai', 'Ġpair', 'Ġe', 'Ġen', 'Ġenc', 'Ġenco', 'Ġencod', 'Ġencodi', 'Ġencodin', 'Ġencoding', 'Ġworks', 'Ġpr', 'Ġpro', 'Ġth', 'Ġthe', 'Ġwa', 'Ġwan', 'Ġwant', 'Ġto', 'ĠT', 'ĠTh', 'ĠThi', 'ĠThis', 'Ġproj', 'Ġproje']


In [17]:
def tokenizeAddText(text):
  pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
  pre_tokenized_text = [word for word, offset in pre_tokenize_result]
  splits = [[l for l in word] for word in pre_tokenized_text]
  for pair, merge in merges.items():
    for idx, split in enumerate(splits):
      i = 0
      while i < len(split) - 1:
        if split[i] == pair[0] and split[i + 1] == pair[1]:
          split = split[:i] + [merge] + split[i + 2 :]
        else:
          i += 1
      splits[idx] = split
  return sum(splits, [])

In [18]:
tokenizeAddText("What a great project to learn byte pair encoding by HuggingFace!")

['W',
 'h',
 'a',
 't',
 'Ġ',
 'a',
 'Ġ',
 'g',
 'r',
 'e',
 'a',
 't',
 'Ġproje',
 'c',
 't',
 'Ġto',
 'Ġlearn',
 'Ġbyte',
 'Ġpair',
 'Ġencoding',
 'Ġby',
 'Ġ',
 'H',
 'u',
 'g',
 'g',
 'i',
 'n',
 'g',
 'F',
 'a',
 'c',
 'e',
 '!']